In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("src")

import torch
import gc
import random

import _dataset
import _prompt
import _mapping
import _util

In [3]:
model, tokenizer = _util.load_OSS()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
# del model, tokenizer
# gc.collect()
# torch.cuda.empty_cache()

In [5]:
random.seed(42)

add_ds = _dataset.create_dataset(num_digits=3, num_samples=20)
print(add_ds)
for add_ds_entry in add_ds:
    base_1_digits = add_ds_entry["base_1_digits"]
    base_2_digits = add_ds_entry["base_2_digits"]
    base_1_num = add_ds_entry["base_1_num"]
    base_2_num = add_ds_entry["base_2_num"]
    source_1_digits = add_ds_entry["source_1_digits"]
    source_2_digits = add_ds_entry["source_2_digits"]
    source_1_num = add_ds_entry["source_1_num"]
    source_2_num = add_ds_entry["source_2_num"]

    # prompt = _prompt.get_stepwise_user_prompt(base_1_num, base_2_num)
    # generated_text = model.generate(tokenizer.encode(prompt, return_tensors="pt").to(model.device), max_new_tokens=200, do_sample=False)
    prompt = _prompt.get_stepwise_prompt(base_1_digits, base_2_digits, base_1_num, base_2_num)
    print(prompt)

[{'base_1_digits': [2, 0, 4], 'base_2_digits': [4, 4, 3], 'base_1_num': 204, 'base_2_num': 443, 'base_sum': 647, 'source_1_digits': [2, 8, 1], 'source_2_digits': [7, 1, 1], 'source_1_num': 281, 'source_2_num': 711, 'source_sum': 992}, {'base_1_digits': [2, 3, 3], 'base_2_digits': [9, 1, 9], 'base_1_num': 233, 'base_2_num': 919, 'base_sum': 1152, 'source_1_digits': [4, 8, 6], 'source_2_digits': [4, 8, 5], 'source_1_num': 486, 'source_2_num': 485, 'source_sum': 971}, {'base_1_digits': [1, 2, 6], 'base_2_digits': [6, 5, 3], 'base_1_num': 126, 'base_2_num': 653, 'base_sum': 779, 'source_1_digits': [4, 5, 1], 'source_2_digits': [2, 7, 2], 'source_1_num': 451, 'source_2_num': 272, 'source_sum': 723}, {'base_1_digits': [6, 5, 9], 'base_2_digits': [5, 1, 8], 'base_1_num': 659, 'base_2_num': 518, 'base_sum': 1177, 'source_1_digits': [9, 1, 6], 'source_2_digits': [2, 9, 5], 'source_1_num': 916, 'source_2_num': 295, 'source_sum': 1211}, {'base_1_digits': [6, 9, 3], 'base_2_digits': [2, 1, 4], 'ba

In [10]:
from torch.nn.utils.rnn import pad_sequence

def print_batch_generation(prompt_template):
    # Prepare all messages for batch processing
    prompts = []
    for add_ds_entry in add_ds:
        base_1_num = add_ds_entry["base_1_num"]
        base_2_num = add_ds_entry["base_2_num"]
        prompt = prompt_template.format(num1=base_1_num, num2=base_2_num)
        prompts.append(prompt)

    # Apply chat template to all messages and create batch
    input_ids = tokenizer(prompts, return_tensors="pt", padding=True, padding_side="left")['input_ids']

    # Generate for entire batch
    generated_texts = model.generate(input_ids.to(model.device), max_new_tokens=300, do_sample=False, pad_token_id=tokenizer.pad_token_id)

    # Decode and print results
    for generated_text in generated_texts:
        print(tokenizer.decode(generated_text))

In [13]:
prompt_template_prefix = '''<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-06-28

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>developer<|message|>'''

reasoning_template_suffix = '''<|end|><|start|>user<|message|>What is {num1}+{num2}?<|end|><|start|>assistant<|channel|>analysis<|message|>'''

simple1 = "# Instructions\nYou are a stepwise adder that reasons concisely. First add the units with carry, then add the next significant digits one at a time. Be sure to approach the problem by adding up digits one at a time in increasing significance. Once reasoning has concluded, give the user only the final answer."

prompt_template1 = prompt_template_prefix + simple1 + reasoning_template_suffix

In [14]:
print_batch_generation(prompt_template1)

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-06-28

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>developer<|message|># Instructions
You are a stepwise adder that reasons concisely. First add the units with carry, then add the next significant digits one at a time. Be sure to approach the problem by adding up digits one at a time in increasing significance. Once reasoning has concluded, give the user only the final answer.<|end|><|start|>user<|message|>What is 204+443?<|end|><|start|>assistant<|channel|>analysis<|message|>The user asks: "What is 204+443?" The developer instructions: "You are a stepwise adder that reasons concisely. First add the units with carry, then add the next significant digits one at a time. Be sure to approach the problem by adding up digits one at a time in increasing significance. Once reasoning has c

In [13]:
for add_ds_entry in add_ds:
    base_1_num = add_ds_entry["base_1_num"]
    base_2_num = add_ds_entry["base_2_num"]

    formatted_prompt = prompt_template5.format(num1=base_1_num, num2=base_2_num)
    
    generated_text = model.generate(tokenizer.encode(formatted_prompt, return_tensors="pt").to(model.device), max_new_tokens=200, do_sample=False)
    print(tokenizer.decode(generated_text[0]).replace(formatted_prompt, ""))

The user asks: "What is 204+432?" The instruction from the developer says: "In analysis, add two numbers stepwise: break the second operand into digits and add to the first operand (e.g. 12 + 34 = 12 + 30 + 4 = 42 + 4 = 46). In final, output only the sum."

So we need to do the addition stepwise: break the second operand (432) into digits: 400 + 30 + 2. Then add to the first operand (204). So 204 + 400 = 604. Then 604 + 30 = 634. Then 634 + 2 = 636. So the sum is 636. In final, output only the sum: "636". The analysis should show the stepwise addition. The final should just output "636". That is the answer.<|end|><|start|>assistant<|channel|>final<|message|>636<|return|>
The user asks: "What is 233+990?" The instruction from the developer says: "In analysis, add two numbers stepwise: break the second operand into digits and add to the first operand (e.g. 12 + 34 = 12 + 30 + 4 = 42 + 4 = 46). In final, output only the sum."

So we need to do the addition stepwise: break 990 into digits:

In [32]:
for add_ds_entry in add_ds:
    base_1_num = add_ds_entry["base_1_num"]
    base_2_num = add_ds_entry["base_2_num"]

    formatted_prompt = prompt_template2.format(num1=base_1_num, num2=base_2_num)
    
    generated_text = model.generate(tokenizer.encode(formatted_prompt, return_tensors="pt").to(model.device), max_new_tokens=200, do_sample=False)
    print(tokenizer.decode(generated_text[0]).replace(formatted_prompt, ""))

We have a conversation. The user asks: "What is 901+945?" The developer instruction says: "In your analysis, add two numbers stepwise. For the final output, output only the sum." So we need to do the addition stepwise in analysis, but final output should be only the sum. So we need to compute 901 + 945. Let's do stepwise: 901 + 945 = 901 + 900 + 45 = 901 + 900 = 1801, then 1801 + 45 = 1846. So the sum is 1846. The final output should be only the sum: "1846". So we output 1846.<|end|><|start|>final<|message|>1846<|return|>
We have a conversation. The user asks: "What is 582+914?" The developer instruction says: "In your analysis, add two numbers stepwise. For the final output, output only the sum." So we need to do the addition stepwise in analysis, but final output should only be the sum. So we need to compute 582 + 914. Let's do stepwise: 582 + 914 = 582 + 900 + 14 = 1482 + 14 = 1496. Wait, 582 + 900 = 1482, plus 14 = 1496. So the sum is 1496. So final output: 1496. But we need to ens

In [35]:
for add_ds_entry in add_ds:
    base_1_num = add_ds_entry["base_1_num"]
    base_2_num = add_ds_entry["base_2_num"]

    formatted_prompt = prompt_template3.format(num1=base_1_num, num2=base_2_num)
    
    generated_text = model.generate(tokenizer.encode(formatted_prompt, return_tensors="pt").to(model.device), max_new_tokens=200, do_sample=False)
    print(tokenizer.decode(generated_text[0]).replace(formatted_prompt, ""))

We have a conversation. The user asks: "What is 901+945?" The developer instruction says: "In the analysis, add two numbers stepwise. For the final output, output only the sum." So we need to do the addition stepwise in the analysis, and then in the final output, just output the sum. So we need to compute 901 + 945. Let's do stepwise: 901 + 945 = 901 + 900 + 45 = 901 + 900 = 1801, then 1801 + 45 = 1846. Alternatively, 901 + 945 = 901 + 900 = 1801, plus 45 = 1846. So the sum is 1846. So final output: 1846.<|end|><|start|>final<|message|>1846<|return|>
We have a conversation. The user asks: "What is 582+914?" The developer instruction says: "In the analysis, add two numbers stepwise. For the final output, output only the sum." So we need to do the addition stepwise in the analysis, and then in the final output, output only the sum. So we need to compute 582 + 914. Let's do stepwise: 582 + 914 = 582 + 900 + 14 = 1482 + 14 = 1496. Or 582 + 914 = 582 + 900 = 1482, plus 14 = 1496. So the sum

In [19]:
for add_ds_entry in add_ds:
    base_1_num = add_ds_entry["base_1_num"]
    base_2_num = add_ds_entry["base_2_num"]

    formatted_prompt = prompt_template5.format(num1=base_1_num, num2=base_2_num)
    
    generated_text = model.generate(tokenizer.encode(formatted_prompt, return_tensors="pt").to(model.device), max_new_tokens=200, do_sample=False)
    print(tokenizer.decode(generated_text[0]).replace(formatted_prompt, ""))

We have a conversation. The user asks: "What is 595+888?" The developer instruction says: "In analysis, add two numbers stepwise. Break the second number into digits first, e.g. 12 + 34 = 12 + 30 + 4. In final, output only the sum."

So we need to do the addition stepwise, breaking the second number into digits. The second number is 888. So we break 888 into 800 + 80 + 8. Then we add 595 + 800 + 80 + 8. We need to do stepwise addition. So 595 + 800 = 1395. Then 1395 + 80 = 1475. Then 1475 + 8 = 1483. So the sum is 1483. The final output should be only the sum: 1483. The analysis should show the stepwise addition. The final should output only the sum
We have a conversation. The user asks: "What is 544+413?" The developer instruction says: "In analysis, add two numbers stepwise. Break the second number into digits first, e.g. 12 + 34 = 12 + 30 + 4. In final, output only the sum."

So we need to do the addition stepwise, breaking the second number into digits. The second number is 413. So

KeyboardInterrupt: 

In [48]:
for add_ds_entry in add_ds:
    base_1_num = add_ds_entry["base_1_num"]
    base_2_num = add_ds_entry["base_2_num"]

    formatted_prompt = prompt_template7.format(num1=base_1_num, num2=base_2_num)
    
    generated_text = model.generate(tokenizer.encode(formatted_prompt, return_tensors="pt").to(model.device), max_new_tokens=200, do_sample=False)
    print(tokenizer.decode(generated_text[0]).replace(formatted_prompt, ""))

We need to compute 901 + 945. The user asks: "What is 901+945?" We need to output the sum. According to the developer instruction: "In analysis, do stepwise addition. In final, output only the sum." So we need to do stepwise addition in analysis, then in final output only the sum. So we need to compute 901 + 945. Let's do stepwise addition: 901 + 945 = 901 + 900 + 45 = 901 + 900 = 1801, plus 45 = 1846. Alternatively, 901 + 945 = 901 + 900 = 1801, plus 45 = 1846. So the sum is 1846. So final output: 1846.<|end|><|start|>final<|message|>1846<|return|>
The user asks: "What is 582+914?" The instruction from the developer says: "In analysis, do stepwise addition. In final, output only the sum." So I need to do stepwise addition in analysis, then output only the sum in final. So I need to compute 582 + 914. Let's do stepwise addition: 582 + 914. 582 + 900 = 1482. 1482 + 14 = 1496. So the sum is 1496. In final, output only the sum: 1496.<|end|><|start|>final<|message|>1496<|return|>
We have a

KeyboardInterrupt: 

In [46]:
print(_mapping.prompt_components.keys())

dict_keys(['system', 'user', 'assistant_header', 'restatement', 'developer_instructions_restatement', 'reasoning_suffix', 'output_header', 'digits_developer', 'digits_reasoning_header', 'digits_reasoning_step', 'prev_carry', 'write_and_carry', 'binary_developer', 'binary_reasoning_header', 'binary_reasoning_step'])
